In [1]:
import os
import pandas as pd
here = os.getcwd()
dimig_folder = os.path.dirname(here)


<br>
<br>
<br>

## 1. AN Debates

First, we get the debates from the AN (and the senate?)

In [2]:
an_debates_folder = os.path.join(dimig_folder, "AN_debats")
corpus_an_debates = os.path.join(an_debates_folder, "data\\corpus_debats_2015_2025.csv")

In [3]:
corpus_an_debates_df = pd.read_csv(corpus_an_debates, sep=",", on_bad_lines='skip')

In [4]:
# for col in corpus_an_debates_df.columns:
#     print(f"Unique values in column '{col}':")
#     print(corpus_an_debates_df[col].unique())

In [5]:
mot_clés = os.path.join(here, "search_terms.json")
mot_clés_dict = pd.read_json(mot_clés, typ='series').to_dict()
# mot_clés_dict

In [6]:
import pandas as pd
import re

def detect_migration_terms(text, nested_dict):
    """
    Parcourt le dictionnaire à 3 niveaux : Thème > Catégorie > Termes
    Utilise les Regex pour éviter les faux positifs et gérer les racines (wildcards).
    """
    if not text:
        return False, None, None, None
    
    text_lower = text.lower()
    
    for theme, categories in nested_dict.items():
        for category, terms in categories.items():
            for term in terms:
                term_lower = term.lower()
                
                # Gestion du wildcard : si le terme finit par *, on adapte la regex
                if term_lower.endswith('*'):
                    root = re.escape(term_lower[:-1])
                    # \b au début pour le mot entier, mais pas à la fin pour laisser la racine libre
                    pattern = rf"\b{root}\w*" 
                else:
                    # Mot exact uniquement grâce aux \b (boundary)
                    pattern = rf"\b{re.escape(term_lower)}\b"
                
                if re.search(pattern, text_lower):
                    return True, term, category, theme
    
    return False, None, None, None

def enrich_corpus_with_detection(df, text_column, search_terms_dict):
    """
    Applique la détection sur le DataFrame et crée les 4 colonnes de résultats.
    """
    # Application de la fonction de détection
    # On utilise zip(*...) pour transformer la liste de tuples en colonnes distinctes
    res = df[text_column].apply(lambda x: detect_migration_terms(x, search_terms_dict))
    
    (df['has_migration_lexicon'], 
     df['matched_keyword'], 
     df['sub_category'], 
     df['broad_theme']) = zip(*res)
    
    return df

# --- EXÉCUTION ---

# Nettoyage préalable des données manquantes
corpus_an_debates_df['content'] = corpus_an_debates_df['content'].fillna('')

print(f"Statistiques avant détection : {corpus_an_debates_df.shape}")

# Lancement de la détection avec le dictionnaire imbriqué
corpus_an_debates_df = enrich_corpus_with_detection(
    corpus_an_debates_df, 
    'content', 
    mot_clés_dict
)

print(f"Statistiques après détection : {corpus_an_debates_df.shape}")



Statistiques avant détection : (3812, 19)
Statistiques après détection : (3812, 23)


In [7]:
# Optionnel : Aperçu des résultats positifs
print(corpus_an_debates_df[corpus_an_debates_df['has_migration_lexicon'] == True].head())

                     id        attachmentUrl     dateSeance   dateParution  \
0  ANSETEXT000053324357      AN_2025-142.pdf  1766448000000  1766534400000   
1  ANSETEXT000053324355  AN_2025-142_Bis.pdf  1766448000000  1766534400000   
3  ANSETEXT000053324356      AN_2025-141.pdf  1766016000000  1766102400000   
5  ANSETEXT000053444518   SENAT_20250125.pdf  1766016000000  1766102400000   
6  ANSETEXT000053314220      AN_2025-140.pdf  1765929600000  1766016000000   

   anneeParution typeAssemblee              origine  legislature    session  \
0           2025            AN  Assemblée nationale           17  2025-2026   
1           2025        AN_BIS  Assemblée nationale           17  2025-2026   
3           2025            AN  Assemblée nationale           17  2025-2026   
5           2025            SN                Sénat           17  2025-2026   
6           2025            AN  Assemblée nationale           17  2025-2026   

  nomSession  ...  attachment_content_length attachment_

In [8]:
corpus_an_debates_df["has_migration_lexicon"].value_counts()

has_migration_lexicon
False    1966
True     1846
Name: count, dtype: int64

In [9]:
corpus_an_debates_df["matched_keyword"].value_counts()

matched_keyword
immigration                  433
migrants                     308
citoyenneté                  286
asile                        130
réfugiés                     111
régularisation               103
expulsion                     70
migration                     62
immigrés                      43
extradition                   33
réfugié                       28
clandestins                   25
migrant                       23
titre de séjour               20
séjour irrégulier             19
clandestin                    15
politique migratoire          14
sans-papiers                  11
CADA                          11
immigré                        9
apatride                       8
réadmission                    6
émigration                     6
FRONTEX                        6
naturalisation                 5
déchéance de nationalité       5
traite des êtres humains       5
crise migratoire               4
MNA                            4
carte de séjour            

In [10]:
# we convert dateSeance from unix to datetime
corpus_an_debates_df['dateSeance'] = (
    pd.to_datetime(corpus_an_debates_df['dateSeance'], unit='ms')
      .dt.strftime('%Y-%m-%dT%H:%M:%SZ')
)
corpus_an_debates_df['dateParution'] = (
    pd.to_datetime(corpus_an_debates_df['dateParution'], unit='ms')
      .dt.strftime('%Y-%m-%dT%H:%M:%SZ')
)
corpus_an_debates_df["title"] = ""

In [11]:
corpus_an_debates_df["content"].iloc[0]

'ASSEMBLÉE \nNATIONALE \n\nJOURNAL OFFICIEL DE LA RÉPUBLIQUE FRANÇAISE \n\nXVIIe Législature  \n\nSESSION ORDINAIRE DE 2025-2026 \n\n101e séance \n\nSéance du mardi 23 décembre 2025 \n\nCompte rendu intégral \n\nLes articles, amendements et annexes figurent dans le fascicule bleu ci-joint \n\nhttp://www.assemblee-nationale.fr      \n\nAnnée 2025. – No 142 A.N. (C.R.) ISSN 2493-2086 Mercredi 24 décembre 2025 \n\n\n\nSOMMAIRE \n\nPRÉSIDENCE DE MME YAËL BRAUN-PIVET \n\n1. Questions au gouvernement (p. 11404) \n\nLOI SPÉCIALE (p. 11404) \n\nM. Boris Vallaud \n\nM. Sébastien Lecornu, Premier ministre \n\nBAISSE DE LA DÉPENSE PUBLIQUE (p. 11405) \n\nM. Jean-Didier Berger \n\nMme Amélie de Montchalin, ministre de l’action et des \ncomptes publics \n\nM. Jean-Didier Berger \n\nCHUTE DU NOMBRE DE RÉGULARISATIONS (p. 11406) \n\nMme Sandrine Rousseau \n\nM. Laurent Nuñez, ministre de l’intérieur \n\nMme Sandrine Rousseau \n\nCONSÉQUENCES DE LA LOI SPÉCIALE  \nPOUR MAYOTTE (p. 11407) \n\nMme Maud 

In [12]:
def extract_titres_loi(content):
    if not content or not isinstance(content, str):
        return None

    titre_start = re.compile(
        r'((?:Projet|Proposition)\s+de\s+loi\b[^(]*)'
    )
    matches = list(titre_start.finditer(content))
    if not matches:
        return None

    titres = []
    seen = set()

    for match in matches:
        raw = match.group(1)
        titre = re.sub(r'\s+', ' ', raw).strip()
        titre = re.sub(r'\s*\(p\.?\s*\d+\).*$', '', titre).strip()
        titre = re.split(r'\s{2,}[A-Z]{4,}', titre)[0].strip()
        titre = titre.rstrip('.,;: ')
        titre = re.split(r'\s+-\s+(?=Projet|Proposition)', titre)[0].strip()
        titre = re.split(r'\s+(?:Texte|Dernier|LPFP|Ensemble)\b', titre)[0].strip()

        if len(titre) < 15:
            continue

        titre_key = re.sub(r'[^a-zàâäéèêëîïôöùûüç\s]', '', titre.lower().strip())
        titre_key = re.sub(r'\s+', ' ', titre_key).strip()

        already_seen = any(
            titre_key in seen_key or seen_key in titre_key
            for seen_key in seen
        )
        if not already_seen:
            seen.add(titre_key)
            titres.append(titre)

    return titres if titres else None


# Application — une seule fois, avec le bon nom
corpus_an_debates_df["titres_loi"] = corpus_an_debates_df["content"].apply(extract_titres_loi)

# Vérification
print(corpus_an_debates_df["titres_loi"].value_counts(dropna=False).head(20))
print(f"\nAvec titre(s): {corpus_an_debates_df['titres_loi'].notna().sum()}")
print(f"Sans titre (None): {corpus_an_debates_df['titres_loi'].isna().sum()}")

corpus_an_debates_df[corpus_an_debates_df["titres_loi"].notna()]["titres_loi"].head(10)

titres_loi
None                                                                                  2434
[Projet de loi de finances pour 2020]                                                   40
[Projet de loi de finances pour 2018]                                                   34
[Projet de loi de finances pour 2017]                                                   31
[Projet de loi de finances pour 2019]                                                   25
[Projet de loi de finances pour 2021]                                                   25
[Projet de loi de finances pour 2022]                                                   21
[Projet de loi de finances rectificative pour 2020]                                     16
[Projet de loi relatif à la bioéthique]                                                 15
[Projet de loi de finances pour 2026]                                                   13
[Projet de loi instituant un système universel de retraite]                    

0     [Projet de loi spéciale prévue par l’article 4...
1     [Projet de loi spéciale prévue par l’article 4...
4     [Proposition de loi portant reconnaissance par...
7     [Projet de loi autorisant l’approbation de la ...
9     [Projet de loi de financement de la sécurité s...
18    [Proposition de loi relative à la protection s...
20    [Proposition de loi organique tendant à modifi...
26    [Proposition de loi portant création d’un stat...
30    [Projet de loi de financement de la sécurité s...
33    [Projet de loi de financement de la sécurité s...
Name: titres_loi, dtype: object

In [13]:
first_none_content = corpus_an_debates_df[corpus_an_debates_df["titres_loi"].isna()]["content"].iloc[0]
print(first_none_content[:5000])

SÉNAT 
JOURNAL OFFICIEL DE LA RÉPUBLIQUE FRANÇAISE 

SESSION ORDINAIRE DE 2025-2026 

COMPTE RENDU INTÉGRAL 

Séance du mardi 23 décembre 2025 

(44e jour de séance de la session)     

Année 2025. – No 126 S. (C.R.) ISSN 2493-1411 Mercredi 24 décembre 2025 



S O M M A I R E  

PRÉSIDENCE DE MME ANNE CHAIN-LARCHÉ 

1. Mise au point au sujet de votes (p. 13342) 

2. Questions orales (p. 13342) 

OBLIGATION DOMICILIAIRE (p. 13342) 

Question no 762 de M. Pierre-Jean Verzelen. – 
Mme Françoise Gatel, ministre de l’aménagement du 
territoire et de la décentralisation. 

EXONÉRATION DE LA TAXE SUR LES SALAIRES AU BÉNÉFICE  
DES ENTREPRISES À BUT D’EMPLOI (p. 13343) 

Question no 772 de M. Simon Uzenat. – Mme Stéphanie 
Rist, ministre de la santé, des familles, de l’autonomie et 
des personnes handicapées ; M. Simon Uzenat. 

MULTIPLICATION DES AGRESSIONS ET DES INTIMIDATIONS  
VISANT LES LIBRAIRIES INDÉPENDANTES (p. 13344) 

Question no 833 de M. Ian Brossat. – Mme Stéphanie Rist, 
minist

<br>
<br>
<br>
<br>
<br>
<br>

## 2. Discours vie publique


In [14]:
discours_folder = os.path.join(dimig_folder, "Collection des discours publics")
corpus_discours = os.path.join(discours_folder, "Collection des discours publics\code\code\src\discours\corpus.json")

In [15]:
corpus_discours_df = pd.read_json(corpus_discours, orient='records')

In [16]:
metadata_expanded = pd.json_normalize(corpus_discours_df['metadata'])
corpus_discours_df = pd.concat([corpus_discours_df.drop('metadata', axis=1), metadata_expanded], axis=1)

print(corpus_discours_df.head())
print("\nColonnes :")
print(corpus_discours_df.columns.tolist())

                                                text  year  \
0  Mesdames et Messieurs, Chers Amis,\nLes turbul...  2015   
1  Emmanuel MACRONBonjour. Comment allez-vous ? J...  2024   
2  M. le président. L'ordre du jour appelle le dé...  2018   
3  Monsieur le Député,\n\n\nNous sommes respectés...  2019   
4  Monsieur le Président,Monsieur le Rapporteur d...  2015   

                                        intervenants  \
0  [{'uid': '/auteur/7323-laurent-fabius', 'str':...   
1  [{'uid': '/auteur/12135-emmanuel-macron', 'str...   
2  [{'uid': '/auteur/8361-delphine-geny-stephann'...   
3  [{'uid': '/auteur/11143-jean-yves-le-drian', '...   
4  [{'uid': '/auteur/6207-carole-delga', 'str': '...   

                                        circonstance     tags_primary  \
0  Déplacement au Maroc les 9 et 10 mars - lancem...  [International]   
1       Déplacement en Guyane les 25 et 26 mars 2024   [Institutions]   
2  Débat sur les conclusions d'un rapport d'infor...       [Économie]  

In [17]:
corpus_discours_df['intervenants'].iloc[3785]

[{'uid': '/auteur/8606-olga-givernet',
  'str': 'Olga Givernet',
  'pos': "Ministre déléguée, chargée de l'énergie"},
 {'uid': '/auteur/286874-oriane-mancini',
  'str': 'Oriane Mancini',
  'pos': 'journaliste'}]

In [18]:
corpus_discours_df.iloc[0], corpus_an_debates_df.iloc[0]

(text              Mesdames et Messieurs, Chers Amis,\nLes turbul...
 year                                                           2015
 intervenants      [{'uid': '/auteur/7323-laurent-fabius', 'str':...
 circonstance      Déplacement au Maroc les 9 et 10 mars - lancem...
 tags_primary                                        [International]
 tags_secondary    [Relations internationales, Relations bilatéra...
 title                                                              
 date                                           2015-03-09T12:00:00Z
 url               https://www.vie-publique.fr/discours/194116-la...
 Name: 0, dtype: object,
 id                                                        ANSETEXT000053324357
 attachmentUrl                                                  AN_2025-142.pdf
 dateSeance                                                2025-12-23T00:00:00Z
 dateParution                                              2025-12-24T00:00:00Z
 anneeParution                    

In [19]:
corpus_discours_df.head()

,text,year,intervenants,circonstance,tags_primary,tags_secondary,title,date,url
0,"Mesdames et Messieurs, Chers Amis,\nLes turbul...",2015,"[{'uid': '/auteur/7323-laurent-fabius', 'str':...",Déplacement au Maroc les 9 et 10 mars - lancem...,[International],"[Relations internationales, Relations bilatéra...",,2015-03-09T12:00:00Z,https://www.vie-publique.fr/discours/194116-la...
1,Emmanuel MACRONBonjour. Comment allez-vous ? J...,2024,"[{'uid': '/auteur/12135-emmanuel-macron', 'str...",Déplacement en Guyane les 25 et 26 mars 2024,[Institutions],"[Collectivités territoriales, Outre-mer, Guyan...",,2024-03-25T12:00:00Z,https://www.vie-publique.fr/discours/293789-em...
2,M. le président. L'ordre du jour appelle le dé...,2018,[{'uid': '/auteur/8361-delphine-geny-stephann'...,Débat sur les conclusions d'un rapport d'infor...,[Économie],"[Vie économique, Politique économique, Politiq...","Déclaration de Mme Delphine Gény-Stéphann, sec...",2018-01-17T12:00:00Z,https://www.vie-publique.fr/discours/204769-de...
3,"Monsieur le Député,\n\n\nNous sommes respectés...",2019,"[{'uid': '/auteur/11143-jean-yves-le-drian', '...",Question au gouvernement à l'Assemblée nationale,[International],"[Asie, Syrie, France - Syrie, Terrorisme, Fran...","Déclaration de M. Jean-Yves Le Drian, ministre...",2019-10-15T12:00:00Z,https://www.vie-publique.fr/discours/271305-je...
4,"Monsieur le Président,Monsieur le Rapporteur d...",2015,"[{'uid': '/auteur/6207-carole-delga', 'str': '...",Première lecture du projet de loi autorisant l...,[International],"[Europe, Moldavie, UE - Moldavie, Relations in...","Déclaration de Mme Carole Delga, secrétaire d'...",2015-04-16T12:00:00Z,https://www.vie-publique.fr/discours/194674-de...


In [20]:
corpus_discours_df['text'] = corpus_discours_df['text'].fillna('')
print(f"Before {corpus_discours_df.shape}")
corpus_discours_df = enrich_corpus_with_detection(corpus_discours_df, 'text', mot_clés_dict)
print(f"After {corpus_discours_df.shape}")

Before (28826, 9)
After (28826, 13)


In [21]:
# let's add adn 'id' column composed of 'VP' + 'url'[37:43]
corpus_discours_df['id'] = 'VP' + corpus_discours_df['url'].str[37:43]

In [22]:
corpus_discours_df["origine"] = "Vie Publique"

In [23]:
legislatures = {
    14: ["2012-06-20T00:00:00Z", "2017-06-20T23:59:59Z"],
    15: ["2017-06-21T00:00:00Z", "2022-06-21T23:59:59Z"],
    16: ["2022-06-22T00:00:00Z", "2024-06-09T23:59:59Z"],
    17: ["2024-07-08T00:00:00Z", "2025-12-31T23:59:59Z"]
}
# we create a column 'legislature' in corpus_discours_df based on dateSeance and the legislatures dict
def assign_legislature(date_str):
    for legislature, (start_date, end_date) in legislatures.items():
        if start_date <= date_str <= end_date:
            return legislature
    return None
corpus_discours_df['legislature'] = corpus_discours_df['date'].apply(assign_legislature)
corpus_discours_df["legislature"].value_counts()



legislature
15.0    12743
14.0     8153
16.0     4317
17.0     2784
Name: count, dtype: int64

<br>
<br>
<br>

# Merge

### Ajout de tags_primary pour les débats AN et Sénats

In [26]:
import re
import ast
import unicodedata
from collections import defaultdict

def parse_tag(val):
    """Parse une valeur de tag (string ou liste)"""
    try:
        if isinstance(val, str):
            return ast.literal_eval(val)
        return val or []
    except:
        return []

def normalize(text):
    """Lowercase + suppression des accents pour matching souple"""
    text = text.lower().strip()
    text = unicodedata.normalize("NFD", text)
    return "".join(c for c in text if unicodedata.category(c) != "Mn")

# Construire le mapping primary <-> set de mots-clés secondary
theme_keywords = defaultdict(set)

for _, row in corpus_discours_df.iterrows():
    primaries = parse_tag(row["tags_primary"])
    secondaries = parse_tag(row["tags_secondary"])
    
    if not primaries or not secondaries:
        continue
    
    for primary in primaries:
        for secondary in secondaries:
            # Décomposer chaque tag secondary en mots individuels significatifs
            # ex: "France - Ukraine" → ["france", "ukraine"]
            # ex: "Protection de l'environnement" → ["protection", "environnement"]
            words = re.findall(r"[a-zàâäéèêëîïôöùûüç]{4,}", normalize(secondary))
            theme_keywords[primary].update(words)

import numpy as np
from collections import Counter

themes = list(theme_keywords.keys())

# Compter dans combien de thèmes apparaît chaque mot
word_theme_count = Counter()
for kws in theme_keywords.values():
    for w in kws:
        word_theme_count[w] += 1

# Ne garder que les mots EXCLUSIFS à 1 seul thème
# (apparaissent dans exactement 1 thème sur 4)
theme_keywords_strict = {}
for theme, kws in theme_keywords.items():
    exclusive = {w for w in kws if word_theme_count[w] == 1 and len(w) >= 5}
    theme_keywords_strict[theme] = exclusive
    print(f"\n{theme}: {len(exclusive)} mots exclusifs")
    print(sorted(exclusive)[:30])


International: 121 mots exclusifs
['acquittement', 'allemand', 'amnesty', 'amnistie', 'andorre', 'antarctique', 'antipersonnel', 'apartheid', 'arabe', 'armements', 'arret', 'asean', 'azerbaidjan', 'bahrein', 'baltes', 'barbade', 'bastia', 'berlusconi', 'bhoutan', 'billet', 'bissau', 'blinde', 'bolivie', 'bosnie', 'botswana', 'burundi', 'cambodge', 'casque', 'cedeao', 'centrafricaine']

Institutions: 92 mots exclusifs
['acfci', 'acquisition', 'afnor', 'anonymat', 'armenien', 'aubry', 'audition', 'autonomiste', 'ayrault', 'badinter', 'banane', 'barnier', 'bendit', 'bordeaux', 'canton', 'centrisme', 'chikungunya', 'cinquieme', 'claude', 'cnfpt', 'cohabitation', 'colmar', 'competences', 'consultative', 'contractuelle', 'contrainte', 'creole', 'daniel', 'decoupage', 'delanoe']

Économie: 90 mots exclusifs
['acnusa', 'annexe', 'aquitaine', 'arcelor', 'augmentation', 'aviculture', 'batellerie', 'belfort', 'besancon', 'biotechnologies', 'boulangerie', 'boursiere', 'cahier', 'ceinture', 'centr

In [27]:
BLACKLIST_INSTITUTIONS = {
    "francais", "francaise", "franc", "republique", "publique",
    "officiel", "jour", "journal", "session", "france",
    "presidence", "residence", "general", "generale", "pour", "dans",
    "projet", "nationale", "national", "gouvernement", "commission",
    "rapport", "politique", "politiques", "article", "proposition",
    "question", "questions", "scrutin", "mise", "mission",
    "contre", "situation", "finance", "finances", "social",
    "maire", "port", "senat", "parti", "lutte",
    "assemblee", "parlement", "parlementaire", "vote", "loi",
    "decret", "amendement", "ministre", "ministere", "service",
    "public", "debat", "seance",
}

# Filtrer les deux niveaux pour Institutions
theme_keywords_strict_filtered = {
    theme: (kws - BLACKLIST_INSTITUTIONS if theme == "Institutions" else kws)
    for theme, kws in theme_keywords_strict.items()  # ← itère sur strict
}


def map_sujets_to_themes_strict_only(sujets_val):
    try:
        sujets_list = parse_tag(sujets_val) if isinstance(sujets_val, str) else (sujets_val or [])
    except:
        return []
    if not sujets_list:
        return []

    sujets_text = normalize(" ".join(sujets_list))
    taxonomy_order = ["Économie", "International", "Institutions", "Société"]
    
    return [
        theme for theme in taxonomy_order
        if any(kw in sujets_text for kw in theme_keywords_strict_filtered.get(theme, set()))
    ]

corpus_an_debates_df["tags_primary"] = corpus_an_debates_df["sujets"].apply(map_sujets_to_themes_strict_only)

print(corpus_an_debates_df["tags_primary"].apply(str).value_counts().head(20))
empty = corpus_an_debates_df["tags_primary"].apply(lambda x: x == []).sum()
print(f"\nNon matchés: {empty} / {len(corpus_an_debates_df)} ({100*empty/len(corpus_an_debates_df):.1f}%)")

tags_primary
[]                                                          1033
['Société']                                                  898
['Économie', 'Société']                                      377
['Institutions', 'Société']                                  277
['Économie']                                                 252
['Institutions']                                             242
['Économie', 'Institutions', 'Société']                      206
['Économie', 'Institutions']                                 114
['International', 'Société']                                 112
['Économie', 'International', 'Institutions', 'Société']      91
['Économie', 'International', 'Société']                      82
['International', 'Institutions', 'Société']                  44
['International']                                             42
['Économie', 'International']                                 16
['International', 'Institutions']                             16
['Économie',

### Intervenants

In [28]:
mandats_file = os.path.join(dimig_folder, "AN_scrutins/Députés_français (Locuteurs)/3_dataset/mandats.csv")
mandats_df = pd.read_csv(mandats_file, sep=",")

C:\Users\rorod\AppData\Local\Temp\ipykernel_28112\3133667031.py:2: DtypeWarning: Columns (20,21,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  mandats_df = pd.read_csv(mandats_file, sep=",")


In [48]:
from collections import defaultdict
import unicodedata
import json
import re

def normalize_name(text):
    """Normalise pour comparaison : minuscules, sans accents, tirets = espaces"""
    text = text.lower().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = text.replace("-", " ")
    return re.sub(r'\s+', ' ', text).strip()

def normalize_name_strict(text):
    """Normalise sans toucher aux tirets — pour le truncated_index"""
    text = text.lower().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return re.sub(r'\s+', ' ', text).strip()

# ======================================================================
# 0. EXTRACTION DES NOMS PROPRES DEPUIS LE JSON BRUT
# ======================================================================

def extract_proper_names(intervenants_val):
    """Parse le JSON brut et extrait les noms propres."""
    try:
        if isinstance(intervenants_val, str):
            items = json.loads(intervenants_val)
        else:
            items = intervenants_val or []
    except:
        return []

    name_pattern = re.compile(
        r'^[A-ZÀ-Ÿ][a-zà-ÿ\-]+(?:\s+(?:de|le|la|les|du|des|d\'|el|El)?\s*[A-ZÀ-Ÿ][a-zà-ÿ\-]+){1,4}-?$'
    )
    civility_split = re.compile(
        r'(?:^|\s)(?:M\.|Mme|Madame|Monsieur)\s+'
    )

    def extract_names_from_text(text):
        if not text:
            return []
        segments = re.split(r'\n{2,}', text)
        results = []
        for segment in segments:
            segment = re.sub(r'\s+', ' ', segment).strip()
            if not segment:
                continue
            splits = re.split(civility_split, segment)
            if len(splits) > 1:
                for part in splits:
                    part = part.strip()
                    if not part:
                        continue
                    match = re.match(
                        r'^([A-ZÀ-Ÿ][a-zà-ÿ\-]+(?:\s+(?:de|le|la|du|des|d\'|el|El)?\s*[A-ZÀ-Ÿ][a-zà-ÿ\-]+){0,4}-?)',
                        part
                    )
                    if match:
                        name = match.group(1).strip()
                        if len(name.split()) >= 2:
                            results.append(name)
            else:
                segment_clean = re.sub(r'^(M\.|Mme|Madame|Monsieur)\s+', '', segment).strip()
                if name_pattern.match(segment_clean):
                    results.append(segment_clean)
                else:
                    matches = re.findall(
                        r'\b([A-ZÀ-Ÿ][a-zà-ÿ\-]+(?:\s+[A-ZÀ-Ÿ][a-zà-ÿ\-]+){1,3}-?)\b',
                        segment_clean
                    )
                    if matches:
                        results.append(max(matches, key=len))
        return results

    seen = set()
    final_results = []

    for item in items:
        for field in ["str", "pos"]:
            raw = item.get(field, "") or ""
            names = extract_names_from_text(raw)
            for name in names:
                if not name:
                    continue
                # Remplacer version tronquée par version complète si disponible
                truncated_match = None
                for seen_name in seen:
                    if name.startswith(seen_name) and len(name) > len(seen_name):
                        truncated_match = seen_name
                        break
                if truncated_match:
                    seen.discard(truncated_match)
                    seen.add(name)
                    for i, r in enumerate(final_results):
                        if r["str"] == truncated_match:
                            final_results[i]["str"] = name
                            break
                elif name not in seen:
                    seen.add(name)
                    final_results.append({"uid": None, "str": name, "pos": ""})
            if names:
                break

    return final_results


# ======================================================================
# 1. CONSTRUIRE L'INDEX DEPUIS mandats_df
# ======================================================================

name_index = {}
prenom_index = defaultdict(list)
truncated_index = defaultdict(list)
nom_seul_index = defaultdict(list)

for _, row in mandats_df.dropna(subset=["ident_prenom_ANM", "ident_nom_ANM"]).drop_duplicates(
    subset=["ident_prenom_ANM", "ident_nom_ANM"]
).iterrows():
    prenom = row["ident_prenom_ANM"].strip()
    nom = row["ident_nom_ANM"].strip()
    acteur_id = row.get("acteur_id", None)
    full_name = f"{prenom} {nom}"
    key = normalize_name(full_name)

    name_index[key] = {"str": full_name, "uid": acteur_id}
    prenom_index[normalize_name(prenom)].append((full_name, acteur_id))
    nom_seul_index[normalize_name(nom)].append((full_name, acteur_id))

    if "-" in nom:
        prefix = nom.split("-")[0] + "-"
        trunc_key = normalize_name_strict(f"{prenom} {prefix}")
        truncated_index[trunc_key].append((full_name, acteur_id))

    if "-" in prenom:
        parts = prenom.split("-")
        for alt_prenom in parts:
            alt_key = normalize_name(f"{alt_prenom} {nom}")
            if alt_key not in name_index:
                name_index[alt_key] = {"str": full_name, "uid": acteur_id}
            if "-" in nom:
                prefix = nom.split("-")[0] + "-"
                alt_trunc_key = normalize_name_strict(f"{alt_prenom} {prefix}")
                if alt_trunc_key not in truncated_index:
                    truncated_index[alt_trunc_key].append((full_name, acteur_id))

print(f"Index : {len(name_index)} noms uniques")
print(f"Prénoms : {len(prenom_index)} prénoms distincts")
print(f"Tronqués : {len(truncated_index)} préfixes indexés")
print(f"Noms seuls : {len(nom_seul_index)} noms distincts")


# ======================================================================
# 2. FONCTION DE RÉSOLUTION
# ======================================================================

def resolve_intervenant(name_str, name_index=name_index,
                         prenom_index=prenom_index,
                         truncated_index=truncated_index,
                         nom_seul_index=nom_seul_index):
    if not name_str:
        return None

    name_norm = normalize_name(name_str)

    # Cas 1 : match exact normalisé (tirets = espaces)
    if name_norm in name_index:
        return name_index[name_norm]

    # Cas 2 : nom tronqué se terminant par "-"
    if name_str.endswith("-") or name_str.endswith("- "):
        trunc_norm = normalize_name_strict(name_str).rstrip("- ").rstrip() + "-"
        for key, candidates in truncated_index.items():
            if key.startswith(trunc_norm):
                if len(candidates) == 1:
                    full, uid = candidates[0]
                    return {"str": full, "uid": uid}
                return None

    # Cas 3 : prénom composé avec espace autour du tiret
    name_clean = re.sub(r'\s*-\s*', '-', name_str).strip()
    name_clean_norm = normalize_name(name_clean)
    if name_clean_norm in name_index:
        return name_index[name_clean_norm]

    # Cas 4 : nom seul ou nom partiel sans prénom
    parts = name_str.strip().split()
    for i in range(len(parts)):
        alt_prenom = " ".join(parts[:i]) if i > 0 else ""
        alt_nom = " ".join(parts[i:])
        alt_nom_norm = normalize_name(alt_nom)
        candidates = [
            (full, uid) for full, uid in nom_seul_index.get(alt_nom_norm, [])
            if not alt_prenom or normalize_name(full).startswith(normalize_name(alt_prenom))
        ]
        if len(candidates) == 1:
            full, uid = candidates[0]
            return {"str": full, "uid": uid}

    return None


# ======================================================================
# 3. APPLIQUER LA RÉSOLUTION SUR LE DATAFRAME
# ======================================================================

def fix_intervenants(intervenants):
    if not isinstance(intervenants, list):
        return intervenants

    fixed = []
    seen = set()

    for item in intervenants:
        name_str = item.get("str", "") or ""

        # Nettoyer les \n et artefacts résiduels
        name_str = re.split(r'\n', name_str)[0].strip()
        name_str = re.sub(r'\s+', ' ', name_str).strip()
        name_str = re.sub(r'\s+(et\s+)?(M\.?|Mme)\s*$', '', name_str).strip()

        if not name_str:
            continue

        resolved = resolve_intervenant(name_str)

        if resolved:
            final_name = resolved["str"]
            uid = resolved["uid"]
        else:
            final_name = name_str
            uid = None

        if final_name and final_name not in seen:
            seen.add(final_name)
            fixed.append({"uid": uid, "str": final_name, "pos": item.get("pos", "")})

    return fixed


corpus_an_debates_df["intervenants"] = (
    corpus_an_debates_df["intervenants"]
    .apply(extract_proper_names)  # parse JSON brut + extrait noms propres
    .apply(fix_intervenants)      # valide/corrige via mandats_df
)


# ======================================================================
# 4. DIAGNOSTIC
# ======================================================================

not_found = [
    item["str"]
    for intervenants in corpus_an_debates_df["intervenants"] if isinstance(intervenants, list)
    for item in intervenants if item["uid"] is None
]

from collections import Counter
print(f"\nIntervenants non résolus: {len(set(not_found))}")
print("\nTop 20 non résolus:")
for name, count in Counter(not_found).most_common(20):
    print(f"  {count:4d}  '{name}'")

total = sum(len(i) for i in corpus_an_debates_df["intervenants"] if isinstance(i, list))
resolved = sum(
    sum(1 for item in i if item["uid"] is not None)
    for i in corpus_an_debates_df["intervenants"] if isinstance(i, list)
)
print(f"\nTaux de résolution: {resolved}/{total} ({100*resolved/total:.1f}%)")

Index : 3881 noms uniques
Prénoms : 703 prénoms distincts
Tronqués : 172 préfixes indexés
Noms seuls : 2758 noms distincts

Intervenants non résolus: 1818

Top 20 non résolus:
   445  'Le Fur'
   196  'La Gontrie'
   129  'Guillaume Gouffier-'
   123  'Benjamin Lucas'
    90  'Da Silva'
    75  'Raymonde Poncet'
    75  'Dominique Estrosi'
    71  'Claire Pitollat'
    71  'Monica Michel'
    63  'Gilles Le'
    56  'Thani Mohamed'
    55  'Les Républicains'
    55  'Charlotte Lecocq'
    52  'Franck Menon'
    51  'Christine Cloarec-'
    49  'Patricia Schil'
    47  'Audrey Dufeu Schubert'
    46  'Noëlle Liene'
    44  'Alexandra Borchio'
    42  'Florence Blatrix'

Taux de résolution: 237949/243245 (97.8%)


In [49]:
corpus_an_debates_df["intervenants"].iloc[0]

[{'uid': 'PA719930', 'str': 'Boris Vallaud', 'pos': ''},
 {'uid': 'PA408578', 'str': 'Jean-Didier Berger', 'pos': ''},
 {'uid': 'PA795076', 'str': 'Sandrine Rousseau', 'pos': ''},
 {'uid': 'PA721486', 'str': 'Christophe Naegelen', 'pos': ''},
 {'uid': 'PA720614', 'str': 'Marine Le Pen', 'pos': ''},
 {'uid': 'PA817211', 'str': 'René Pilato', 'pos': ''},
 {'uid': 'PA721670', 'str': 'Amélie de Montchalin', 'pos': ''},
 {'uid': 'PA720728', 'str': 'Jean-Paul Mattei', 'pos': ''},
 {'uid': 'PA842271', 'str': 'Emmanuel Maurel', 'pos': ''},
 {'uid': 'PA795778', 'str': 'Jean-Philippe Tanguy', 'pos': ''},
 {'uid': 'PA841251', 'str': 'Mathilde Feld', 'pos': ''},
 {'uid': 'PA793094', 'str': 'Nicolas Ray', 'pos': ''},
 {'uid': 'PA722162', 'str': 'Nicolas Turquois', 'pos': ''},
 {'uid': 'PA719558', 'str': 'Erwan Balanant', 'pos': ''},
 {'uid': 'PA718850', 'str': 'Pierre Cordier', 'pos': ''},
 {'uid': 'PA642847', 'str': 'Thibault Bazin', 'pos': ''},
 {'uid': 'PA643210', 'str': 'Sébastien Lecornu', 'po

<br>
<br>


Merge final et enregistrement




In [57]:
# keys are an debates columns, values are discours columns
# keys are an debates columns, values are discours columns
corresponding_columns = {
    'content': 'text',
    'dateSeance': 'date',
    'anneeParution': 'year',
    'attachmentUrl': 'source',
}
corpus_an_debates_df = corpus_an_debates_df.rename(columns=corresponding_columns)

In [58]:
corpus_an_debates_df.columns.to_list()

['id',
 'source',
 'date',
 'dateParution',
 'year',
 'typeAssemblee',
 'origine',
 'legislature',
 'session',
 'nomSession',
 'pathToFile',
 'attachment_title',
 'attachment_name',
 'attachment_content_length',
 'attachment_content_type',
 'titre_seance',
 'sujets',
 'intervenants',
 'text',
 'has_migration_lexicon',
 'matched_keyword',
 'sub_category',
 'broad_theme',
 'title',
 'titres_loi',
 'tags_primary']

In [59]:
corpus_discours_df.columns.to_list()

['text',
 'year',
 'intervenants',
 'circonstance',
 'tags_primary',
 'tags_secondary',
 'title',
 'date',
 'url',
 'has_migration_lexicon',
 'matched_keyword',
 'sub_category',
 'broad_theme',
 'id',
 'origine',
 'legislature']

In [62]:
columns_to_keep = [
    'id',
    'date',
    'year',
    'origine',
    'legislature',
    'title',
    'text',
    'intervenants',
    'tags_primary',
    'has_migration_lexicon',
    'matched_keyword',
    'sub_category',
    'broad_theme',
]

corpus_an_debates_df = corpus_an_debates_df[columns_to_keep]
corpus_discours_df = corpus_discours_df[columns_to_keep]

In [63]:
corpus_migration_df = pd.concat([corpus_an_debates_df[corpus_an_debates_df['has_migration_lexicon'] == True], corpus_discours_df[corpus_discours_df['has_migration_lexicon'] == True]], ignore_index=True)

In [65]:
import zipfile

# Rename to be consistent w/ the google docs
csv_path = os.path.join(here, "corpus_migration.csv")
zip_path = os.path.join(here, "corpus_migration.zip")

corpus_migration_df.to_csv(csv_path, index=False)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, "corpus_migration.csv")

In [ ]:
def proportion_positifs(df, by="year", date_col="date"):
    df = df.copy()
    
    # conversion de la date ISO
    df[date_col] = pd.to_datetime(df[date_col], utc=True)

    if by == "year":
        df["period"] = df[date_col].dt.year
    elif by == "month":
        df["period"] = df[date_col].dt.to_period("M")
    else:
        raise ValueError("by doit être 'year' ou 'month'")

    result = (
        df.groupby("period")['found']
        .mean()
        .reset_index(name="prop_positive")
    )

    return result

In [ ]:
import matplotlib.pyplot as plt

def plot_proportions(df_prop, title="Proportion de textes positifs"):
    """
    Plot des proportions de textes positifs dans le temps.

    Parameters
    ----------
    df_prop : DataFrame
        Résultat de proportion_positifs (colonnes: period, prop_positive)
    title : str
        Titre du graphique
    """

    df_prop = df_prop.copy()

    # conversion pour assurer un bon affichage temporel
    df_prop["period"] = df_prop["period"].astype(str)

    plt.figure(figsize=(10,5))
    plt.plot(df_prop["period"], df_prop["prop_positive"], marker="o")

    plt.xlabel("Période")
    plt.ylabel("Proportion de textes positifs")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show()

In [ ]:
plot_proportions(proportion_positifs(corpus_an_debates_df, by="year", date_col="dateSeance"), title="Proportion de textes positifs dans le corpus par année")


KeyError: 'dateSeance'

In [ ]:
res = proportion_positifs(date_col="dateSeance",
    df = corpus_an_debates_df[
        pd.to_datetime(corpus_an_debates_df["dateSeance"]) < "2020-12-23"
    ],
    by="month"
)
plot_proportions(res, title="Proportion de textes positifs dans les débats de l'AN et du Sénat par Année-Mois")

In [ ]:
proportion_positifs(corpus_discours_df, by="year", date_col="date")
proportion_positifs(corpus_discours_df, by="month", date_col="date")